In [ ]:
import json
import os
import shutil
from datetime import datetime, timezone

from kaggle.api.kaggle_api_extended import KaggleApi

# =========================================================
# 설정
# =========================================================

# all / active / completed
COMPETITION_STATE = "all"

# latestDeadline
# earliestDeadline
# recentlyCreated
# numberOfTeams
# prize
SORT_BY = "latestDeadline"

PAGE_SIZE = 100

# =========================================================
# 현재 실행 파일 위치
# =========================================================

BASE_DIR = os.getcwd()

# kaggle.json 위치
json_path = os.path.join(BASE_DIR, "kaggle.json")

# Kaggle API가 요구하는 기본 경로
kaggle_dir = os.path.join(os.path.expanduser("~"), ".kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

# kaggle.json 복사
target_json = os.path.join(kaggle_dir, "kaggle.json")
shutil.copy(json_path, target_json)

# =========================================================
# API 인증
# =========================================================

api = KaggleApi()
api.authenticate()

# =========================================================
# 전체 competition 수집
# =========================================================

results = []
allResult = []
page = 1

while True:

    print(f"\n===== PAGE {page} =====")

    # 응답 객체
    response = api.competitions_list(
        page=page,
        page_size=PAGE_SIZE,
        sort_by=SORT_BY
    )

    # 실제 competition 리스트
    competitions = response.competitions

    print("현재 페이지 개수:", len(competitions))

    # 더 이상 없으면 종료
    if len(competitions) == 0:
        break

    for comp in competitions:

        try:
            dicComp = comp.to_dict()
            results.append({
                "title": dicComp.get("title"),
                "ref": dicComp.get("ref"),
                "category": dicComp.get("category"),
                "teams": dicComp.get("teamCount"),
                "reward": dicComp.get("reward"),
                "enabledDate": str(dicComp.get("enabledDate")),
                "deadline": str(dicComp.get("deadline")),
                "url": dicComp.get("url"),
                "description": dicComp.get("description"),
                "organizationName": dicComp.get("organizationName"),
            })

            print("=" * 50)
            print("제목:", comp.title)
            print("카테고리:", comp.category)
            print("참여팀", dicComp.get("teamCount"))
            print("보상:", comp.reward)
            print("시작일", dicComp.get("enabledDate"))
            print("마감일:", comp.deadline)
            print("URL:", comp.url)
            allResult.append(dicComp)
        except Exception as e:

            print("실패:", comp.ref, e)

    page += 1

# =========================================================
# 결과 저장
# =========================================================

with open("all_competitions.json", "w", encoding="utf-8") as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )
with open("all_competitions_detail.json", "w", encoding="utf-8") as f:

    json.dump(
        allResult,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n========================")
print("최종 개수:", len(results))
print("========================")

In [ ]:
allResult[0]["teamCount"]

In [ ]:
allResult.sort(
    key=lambda x: x.get("teamCount") if x.get("teamCount") else 0,
    reverse=True
)